In [36]:
import duckdb

In [37]:
connection = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [38]:
df = connection.execute("""
                        
                    SELECT * FROM (    
                                            SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestão DESC) row_num
                                            FROM bronze_z0019
                                            where data_ingestão >= '2026-05-23'
                                 )
                        WHERE row_num = 1

                        """).fetchdf()
df

,NATBR,MAKTX,WERKS,MAINS,LABST,arquivo,data_ingestão,row_num
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-05-23 20:29:52.084393,1
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-05-23 20:29:52.084393,1
2,10003,PREGO,BT10,100,60,z0019_2.csv,2026-05-23 20:30:10.387842,1
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-05-23 20:30:10.387842,1
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-05-23 20:30:10.387842,1


In [39]:
df_final = df.drop(columns=['row_num', 'data_ingestão', 'arquivo'])

df_final = df_final.rename(columns={'NATBR': 'id', 'MAKTX': 'nm_produto', 'WERKS': 'id_categoria', 'MAINS': 'id_fornecedor', 'LABST': 'vl_preco'})

df_final

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10003,PREGO,BT10,100,60
3,10004,SERRA,BT50,100,200
4,10005,MACHADO,BT50,100,100


In [42]:
df2.dtypes

id                 int64
nm_produto        string
id_categoria      string
id_fornecedor      int64
vl_preco         float64
dtype: object

In [41]:
df2 = df_final
df2 = df2.astype({
                  'id': 'int64',
                  'nm_produto': 'string',
                  'id_categoria': 'string',
                  'id_fornecedor': 'int64',
                  'vl_preco': 'float64'
                })


In [43]:
connection.execute("""
                    CREATE TABLE IF NOT EXISTS produtos (
                        id BIGINT,
                        nm_produto TEXT,
                        id_categoria TEXT,
                        id_fornecedor BIGINT,
                        vl_preco FLOAT
                    )
                """)

In [44]:
df2

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10003,PREGO,BT10,100,60.0
3,10004,SERRA,BT50,100,200.0
4,10005,MACHADO,BT50,100,100.0


In [47]:
connection.execute(""" insert into produtos select * from df2""")

In [48]:
df_result = connection.execute("""
                            SELECT * FROM produtos
                        """).fetchdf()
df_result

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10003,PREGO,BT10,100,60.0
3,10004,SERRA,BT50,100,200.0
4,10005,MACHADO,BT50,100,100.0


In [49]:
connection.close()